# Notebook 17b — Realised-FLOP calibration and checkpoint provenance repair

**Why this notebook exists.** The committed `structured_screening_results_calibrated.csv`
is scientifically useful, but the branch does not contain the code that created the
binary-search prefix calibration or the calibrated checkpoints. This notebook makes that
step reproducible.

It does four things:

1. calibrates each pruning prefix against **actual profiled dense FLOPs**;
2. saves the exact removed groups, prefix-search trace, surgery audit, and checkpoint;
3. records complete pre/post recovery semantic metrics, not macro-F1 alone;
4. creates a model registry consumed by the one-shot test audit.

This is a provenance repair. It does **not** redefine the archived screening result and it
does not use the test split.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, platform, hashlib
import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

candidates = [
    os.environ.get("SABER_REPO"),
    "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression",
    str(Path.cwd()),
]
REPO = None
for candidate in candidates:
    if not candidate:
        continue
    p = Path(candidate).expanduser()
    if (p / "src/saber").is_dir() and (p / "config").is_dir():
        REPO = p.resolve()
        break
if REPO is None:
    raise FileNotFoundError("Set SABER_REPO to the saber-ids-method repository.")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Repository:", REPO)
print("Device:", DEVICE)
print("Python:", sys.version.split()[0], "|", platform.platform())


In [ ]:
import yaml
from copy import deepcopy
from torch import nn

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate
from src.saber.adapters import collect_logits, file_sha256, config_hash
from src.saber.surgery import (
    prune_cnn1d_channels, profile_forward_flops, count_parameters,
)
from src.saber.postg5 import calibrate_prefix_count

TRAIN_LOADER, VAL_LOADER, TEST_LOADER, TEACHER, CLASS_NAMES = load_bridge()
TEACHER = TEACHER.to(DEVICE).eval()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)

with open(REPO / "config/saber.yaml", "r", encoding="utf-8") as handle:
    CFG = yaml.safe_load(handle)
MIN_W = int(CFG["groups"]["minimum_remaining_per_layer"])
TARGETS = [float(v) for v in CFG["structured_pruning"]["flops_reduction_budgets"]]
TOLERANCE = 0.015
OUT = REPO / "results/saber/17b_calibrated_checkpoint_freeze"
OUT.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(REPO / "results/saber/15_baselines/group_baseline_scores.csv")
v2 = pd.read_csv(REPO / "results/saber/16b_score_v2/v2_scores.csv")
scores = scores.merge(v2[["group_id", "v_c"]], on="group_id", how="left")
if scores["v_c"].isna().any():
    raise RuntimeError("V-C is missing for one or more groups.")

METHOD_SCORE = {
    "random": "random",
    "magnitude": "magnitude",
    "taylor": "taylor",
    "fisher": "fisher",
    "saber_v2": "v_c",
}
robust_graph = pd.read_csv(REPO / "results/saber/14_risk_graph/asvg_edges_robust.csv")
teacher_logits, val_labels, _ = collect_logits(TEACHER, VAL_LOADER, device=DEVICE)
teacher_audit = full_model_audit(teacher_logits, val_labels, taxonomy, DEFAULT_COST_PROFILES)

example = next(iter(VAL_LOADER))[0][:8].to(DEVICE)
m0_flops = profile_forward_flops(TEACHER, example)["flops_per_item"]
print("M0 validation macro-F1:", teacher_audit["fine_macro_f1"])
print("M0 FLOPs/item:", m0_flops)


In [ ]:
# Shared class weights and deterministic full-recovery training protocol.
train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
counts = np.bincount(train_y, minlength=len(CLASS_NAMES))
weights = np.zeros_like(counts, dtype=float)
present = counts > 0
weights[present] = 1.0 / np.sqrt(counts[present])
weights[present] /= weights[present].mean()
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def evaluate(model):
    logits, labels, _ = collect_logits(model, VAL_LOADER, device=DEVICE)
    audit = full_model_audit(logits, labels, taxonomy, DEFAULT_COST_PROFILES)
    awbir, _ = action_weighted_boundary_inversion_rate(
        teacher_logits, logits, labels, robust_graph
    )
    audit["awbir"] = float(awbir)
    return logits, labels, audit

def full_recovery(model, epochs=8, lr=1e-3, patience=3):
    # Archive-compatible checkpoint rule: best fine macro-F1.
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    best_state = deepcopy(model.state_dict())
    best_f1 = -np.inf
    stale = 0
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        total = 0.0
        seen = 0
        for x, y in TRAIN_LOADER:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            z = model(x)
            loss = criterion(z, y)
            loss.backward()
            optimizer.step()
            total += float(loss.detach().cpu()) * len(y)
            seen += len(y)
        _, _, audit = evaluate(model)
        row = {
            "epoch": epoch,
            "train_loss": total / max(seen, 1),
            **{f"val_{k}": float(v) for k, v in audit.items() if np.isscalar(v)},
        }
        history.append(row)
        if audit["fine_macro_f1"] > best_f1 + 1e-6:
            best_f1 = float(audit["fine_macro_f1"])
            best_state = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break
    model.load_state_dict(best_state)
    return model.eval(), pd.DataFrame(history)

def valid_order(score_column):
    table = scores.sort_values(score_column, ascending=True)
    remaining = table.groupby("module_path")["group_id"].count().to_dict()
    order = []
    for row in table.itertuples():
        layer = str(row.module_path)
        if remaining[layer] - 1 < MIN_W:
            continue
        remaining[layer] -= 1
        order.append((layer, int(row.channel_index), str(row.group_id)))
    return order

def prune_from_prefix(order, k):
    prune_map = {}
    for layer, channel, _ in order[:int(k)]:
        prune_map.setdefault(layer, []).append(channel)
    prune_map = {layer: sorted(channels) for layer, channels in prune_map.items()}
    student, audit = prune_cnn1d_channels(
        TEACHER, prune_map, example, minimum_remaining_per_layer=MIN_W
    )
    return student.to(DEVICE), audit, prune_map

def realised_reduction(model):
    f = profile_forward_flops(model, example)["flops_per_item"]
    return 1.0 - float(f) / float(m0_flops)


In [ ]:
registry_path = OUT / "shallow_frozen_model_registry.csv"
registry_rows = pd.read_csv(registry_path).to_dict("records") if registry_path.exists() else []
done = {(str(r["method"]), float(r["target_flops"])) for r in registry_rows}

archived_path = REPO / "results/saber/17_structured_selection/structured_screening_results_calibrated.csv"
archived = pd.read_csv(archived_path) if archived_path.exists() else pd.DataFrame()

for method, score_column in METHOD_SCORE.items():
    order = valid_order(score_column)
    for target in TARGETS:
        key = (method, target)
        if key in done:
            continue
        print(f"\n=== {method} target={target:.0%} ===")
        calibration = calibrate_prefix_count(
            order,
            target_reduction=target,
            build_model=lambda k: prune_from_prefix(order, k)[0],
            realised_reduction=realised_reduction,
            tolerance=TOLERANCE,
        )
        k = calibration.prefix_length
        student, surgery_audit, prune_map = prune_from_prefix(order, k)
        pre_logits, _, pre = evaluate(student)
        student, history = full_recovery(student)
        post_logits, _, post = evaluate(student)

        tag = f"{method}_r{int(round(target * 100)):02d}cal"
        removed = pd.DataFrame(
            [{"module_path": layer, "channel_index": channel, "group_id": gid}
             for layer, channel, gid in order[:k]]
        )
        removed.to_csv(OUT / f"{tag}_removed_groups.csv", index=False)
        pd.DataFrame(calibration.trace).to_csv(OUT / f"{tag}_prefix_search.csv", index=False)
        surgery_audit.to_csv(OUT / f"{tag}_surgery_audit.csv", index=False)
        history.to_csv(OUT / f"{tag}_finetune_history.csv", index=False)

        checkpoint = OUT / f"{tag}_checkpoint.pt"
        torch.save(
            {
                "state_dict": student.cpu().state_dict(),
                "architecture": "CNN1D-2block",
                "method": method,
                "score_column": score_column,
                "target_flops": target,
                "realised_flops": calibration.realised_reduction,
                "prefix_length": k,
                "prune_map": prune_map,
                "class_names": CLASS_NAMES,
            },
            checkpoint,
        )
        student = student.to(DEVICE)

        row = {
            "architecture": "CNN1D-2block",
            "method": method,
            "score_column": score_column,
            "target_flops": target,
            "realised_flops": calibration.realised_reduction,
            "flops_absolute_error": calibration.absolute_error,
            "within_1p5_points": calibration.within_tolerance,
            "prefix_length": k,
            "parameters": count_parameters(student),
            "checkpoint": str(checkpoint.relative_to(REPO)),
            "checkpoint_sha256": file_sha256(checkpoint),
            "removed_groups": str((OUT / f"{tag}_removed_groups.csv").relative_to(REPO)),
            **{f"pre_{name}": float(value) for name, value in pre.items() if np.isscalar(value)},
            **{f"post_{name}": float(value) for name, value in post.items() if np.isscalar(value)},
        }
        if not archived.empty:
            old = archived[(archived["method"] == method) & np.isclose(archived["budget"], target)]
            if len(old):
                old = old.iloc[0]
                row["archive_macro_f1_difference"] = row["post_fine_macro_f1"] - float(old["post_macro_f1"])
                row["archive_awbir_difference"] = row["post_awbir"] - float(old["post_awbir"])
        registry_rows.append(row)
        pd.DataFrame(registry_rows).to_csv(registry_path, index=False)
        print({k: row[k] for k in ["realised_flops", "post_fine_macro_f1", "post_family_macro_f1", "post_awbir"]})

registry = pd.DataFrame(registry_rows).sort_values(["target_flops", "method"])
display(registry)


In [ ]:
# Provenance audit. This must pass before Notebook 21 is allowed to touch test data.
required = {
    "architecture", "method", "target_flops", "realised_flops", "checkpoint",
    "checkpoint_sha256", "removed_groups", "post_fine_macro_f1",
    "post_family_macro_f1", "post_awbir", "post_hsr_balanced_soc",
}
missing = required.difference(registry.columns)
if missing:
    raise RuntimeError(f"Registry is incomplete: {sorted(missing)}")
if len(registry) != len(METHOD_SCORE) * len(TARGETS):
    raise RuntimeError(f"Expected {len(METHOD_SCORE) * len(TARGETS)} rows, found {len(registry)}")
if not registry["within_1p5_points"].all():
    raise RuntimeError("At least one operating point misses the realised-FLOP tolerance.")
for row in registry.itertuples():
    path = REPO / row.checkpoint
    if not path.exists() or file_sha256(path) != row.checkpoint_sha256:
        raise RuntimeError(f"Checkpoint integrity failure: {path}")

audit = {
    "notebook": "17b_realized_flop_calibration_and_checkpoint_freeze.ipynb",
    "n_models": int(len(registry)),
    "max_abs_flop_error": float(registry["flops_absolute_error"].max()),
    "config_hash": config_hash(CFG),
    "status": "ready_for_test_registry",
}
(OUT / "registry_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(json.dumps(audit, indent=2))
